<a href="https://colab.research.google.com/github/ayushi777lodhi-stack/RAG/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
!nvidia-smi

Wed Jul  8 12:12:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   65C    P0             32W /   70W |    1545MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [37]:
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
12.1
Tesla T4


In [38]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [39]:
import os

if "COLAB_GPU" in os.environ:
  print("installing req")
  !pip install PyMuPDF
  !pip install tqdm
  !pip install accelerate
  !pip install bitsandbytes
  !pip install flash-attn --no-build-isolation

installing req


In [40]:
!pip uninstall -y torch torchvision transformers sentence-transformers
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -U transformers sentence-transformers

Found existing installation: torch 2.5.1+cu121
Uninstalling torch-2.5.1+cu121:
  Successfully uninstalled torch-2.5.1+cu121
Found existing installation: torchvision 0.20.1+cu121
Uninstalling torchvision-0.20.1+cu121:
  Successfully uninstalled torchvision-0.20.1+cu121
Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
Found existing installation: sentence-transformers 3.0.1
Uninstalling sentence-transformers-3.0.1:
  Successfully uninstalled sentence-transformers-3.0.1
Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp312-cp312-linux_x86_64.whl (780.4 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp312-cp312-linux_x86_64.whl (7.3 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the follow

  Using cached transformers-5.13.0-py3-none-any.whl.metadata (32 kB)
  Using cached sentence_transformers-5.6.0-py3-none-any.whl.metadata (18 kB)
  Using cached huggingface_hub-1.22.0-py3-none-any.whl.metadata (14 kB)
Using cached transformers-5.13.0-py3-none-any.whl (11.5 MB)
Using cached sentence_transformers-5.6.0-py3-none-any.whl (596 kB)
Using cached huggingface_hub-1.22.0-py3-none-any.whl (765 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2


In [41]:
from google.colab import files

uploaded=files.upload()

pdf_path=list(uploaded.keys())[0]
print("uploaded pdf:",pdf_path)



Saving Human-Nutrition-2020-Edition-1598491699.pdf to Human-Nutrition-2020-Edition-1598491699 (3).pdf
uploaded pdf: Human-Nutrition-2020-Edition-1598491699 (3).pdf


In [42]:
import os

print("pdf_path:", pdf_path)
print("Exists:", os.path.exists(pdf_path))

pdf_path: Human-Nutrition-2020-Edition-1598491699 (3).pdf
Exists: True


In [43]:
import fitz

doc = fitz.open(pdf_path)

print("Page count:", doc.page_count)
print("Metadata:", doc.metadata)

Page count: 1208
Metadata: {'format': 'PDF 1.7', 'title': 'Human Nutrition: 2020 Edition', 'author': '', 'subject': '', 'keywords': '', 'creator': 'Pressbooks 5.9.2', 'producer': 'Prince 12.5 (www.princexml.com)', 'creationDate': '', 'modDate': '', 'trapped': '', 'encryption': None}


In [45]:
import fitz
from tqdm.auto import tqdm

def text_formatting(text: str)->str:
  cleaned_text=text.replace("\n"," ").strip()
  return cleaned_text


def open_and_read_pdf(pdf_path:str)->list[dict]:
  doc=fitz.open(pdf_path)
  pages_and_texts=[]
  for page_number, page in tqdm(enumerate(doc)):
    text=page.get_text()
    text=text_formatting(text)
    pages_and_texts.append({"page_number":page_number - 41,
                            "page_char_count":len(text),
                            "page_word_count":len(text.split(" ")),
                            "page_sentence_count_raw":len(text.split(". ")),
                            "page_token_count":len(text)/4,
                            "text":text})

  return pages_and_texts

pages_and_texts=open_and_read_pdf(pdf_path=pdf_path)
pages_and_texts[:2]

0it [00:00, ?it/s]

[{'page_number': -41,
  'page_char_count': 29,
  'page_word_count': 4,
  'page_sentence_count_raw': 1,
  'page_token_count': 7.25,
  'text': 'Human Nutrition: 2020 Edition'},
 {'page_number': -40,
  'page_char_count': 0,
  'page_word_count': 1,
  'page_sentence_count_raw': 1,
  'page_token_count': 0.0,
  'text': ''}]

In [46]:
print(len(pages_and_texts))

1208


In [47]:
import random
random.sample(pages_and_texts,k=3)

[{'page_number': 470,
  'page_char_count': 832,
  'page_word_count': 139,
  'page_sentence_count_raw': 5,
  'page_token_count': 208.0,
  'text': 'Learning Activities  Technology Note: The second edition of the Human  Nutrition Open Educational Resource (OER) textbook  features interactive learning activities.\xa0 These activities are  available in the web-based textbook and not available in the  downloadable versions (EPUB, Digital PDF, Print_PDF, or  Open Document).  Learning activities may be used across various mobile  devices, however, for the best user experience it is strongly  recommended that users complete these activities using a  desktop or laptop computer and in Google Chrome.  \xa0 An interactive or media element has been  excluded from this version of the text. You can  view it online here:  http://pressbooks.oer.hawaii.edu/ humannutrition2/?p=301  \xa0 An interactive or media element has been  excluded from this version of the text. You can  470  |  The Atom'},
 {'page_n

In [92]:
from spacy.lang.en import English

nlp=English()

nlp.add_pipe("sentencizer")

In [50]:
for item in tqdm(pages_and_texts):
  item["sentences"]=list(nlp(item["text"]).sents)

  item["sentences"]=[str(sentence) for sentence in item["sentences"]]

  item["page_sentence_count_spacy"]=len(item["sentences"])

  0%|          | 0/1208 [00:00<?, ?it/s]

In [52]:
slice_size=10

def split_list(input_list:list, slice_size:int)->list[list[str]]:
  return [input_list[i:i + slice_size] for i in range(0, len(input_list), slice_size)]

for item in tqdm(pages_and_texts):
  item["sentence_chunks"]=split_list(input_list=item["sentences"], slice_size=slice_size)
  item["num_chunks"]=len(item["sentence_chunks"])

  0%|          | 0/1208 [00:00<?, ?it/s]

In [53]:
random.sample(pages_and_texts,k=1)

[{'page_number': 443,
  'page_char_count': 1822,
  'page_word_count': 308,
  'page_sentence_count_raw': 17,
  'page_token_count': 455.5,
  'text': 'Effects of Alcohol Abuse on the Brain  A small amount (up to 10%) of the liver acetaldehyde may  accumulate inside the liver cells. As more alcohol is ingested, this  stimulates the production of acetaldehyde by both the alcohol  dehydrogenase and MEOS systems. As the levels of acetaldehyde  increase inside the liver cells with heavy consumption of alcohol,  some of the acetaldehyde diffuse into the blood circulation. In  circulation, high levels of acetaldehyde cause nausea and vomiting.  Vomiting causes more body dehydration and loss of electrolytes.  If the dehydration becomes severe enough, this can impair brain  function and a person may lose consciousness.  Alcohol can adversely affect nearly every area of the brain. When  BAC rises, the central nervous system is depressed. Alcohol disrupts  the way nerve cells communicate with each o

In [54]:
import re

pages_and_chunks=[]
for item in tqdm(pages_and_texts):
  for sentence_chunk in item["sentence_chunks"]:
    chunk_dict={}
    chunk_dict["page_number"]=item["page_number"]
    joined_sentence_chunk="".join(sentence_chunk).replace("  "," ").strip()
    joined_sentence_chunk=re.sub(r'\.([A-Z])',r'.\1',joined_sentence_chunk)
    chunk_dict["sentence_chunk"]=joined_sentence_chunk

    chunk_dict["chunk_char_count"]=len(joined_sentence_chunk)
    chunk_dict["chunk_word_count"]=len(joined_sentence_chunk.split(" "))
    chunk_dict["chunk_token_count"]=len(joined_sentence_chunk)/4

    pages_and_chunks.append(chunk_dict)

len(pages_and_chunks)


  0%|          | 0/1208 [00:00<?, ?it/s]

1843

In [55]:
random.sample(pages_and_chunks,k=1)

[{'page_number': 244,
  'sentence_chunk': 'your body senses blood glucose levels and maintains the glucose “temperature” in the target range.The glucose thermostat is located within the cells of the pancreas.After eating a meal containing carbohydrates glucose levels rise in the blood. Insulin-secreting cells in the pancreas sense the increase in blood glucose and release the hormone, insulin, into the blood.Insulin sends a signal to the body’s cells to remove glucose from the blood by transporting it into different organ cells around the body and using it to make energy.In the case of muscle tissue and the liver, insulin sends the biological message to store glucose away as glycogen.The presence of insulin in the blood signifies to the body that glucose is available for fuel.As glucose is transported into the cells around the body, the blood glucose levels decrease.Insulin has an opposing hormone called glucagon.Glucagon-secreting cells in the pancreas sense the drop in glucose and, i

In [56]:
import pandas as pd
df=pd.DataFrame(pages_and_chunks)
df.describe()

,page_number,chunk_char_count,chunk_word_count,chunk_token_count
count,1843.000000,1843.000000,1843.000000,1843.000000
mean,583.381443,731.114487,109.004883,182.778622
std,347.788670,445.651036,69.335175,111.412759
min,-41.000000,12.000000,3.000000,3.000000
25%,280.500000,313.500000,43.000000,78.375000
50%,586.000000,745.000000,111.000000,186.250000
75%,890.000000,1112.000000,168.000000,278.000000
max,1166.000000,1824.000000,290.000000,456.000000


In [58]:
min_token_length=30
for row in df[df["chunk_token_count"]<=min_token_length].sample(5).iterrows():
  print(f"chunk token count: {row[1]["chunk_token_count"]} | text {row[1]["sentence_chunk"]}")

chunk token count: 23.5 | text view it online here: http://pressbooks.oer.hawaii.edu/ humannutrition2/?p=153   194 | Chloride
chunk token count: 24.5 | text U.S. Food and Drug Administration. https://www.fda.gov/food/ 1030 | The Effect of New Technologies
chunk token count: 20.25 | text http://pressbooks.oer.hawaii.edu/ humannutrition2/?p=507   Sports Nutrition | 971
chunk token count: 28.0 | text Fluid balance refers to maintaining the distribution of water in the body. 386 | Protein’s Functions in the Body
chunk token count: 11.0 | text Accessed October 5, 2017. Introduction | 433


In [59]:
pages_and_chunks_over_min_length=df[df["chunk_token_count"]>min_token_length].to_dict(orient="records")
pages_and_chunks_over_min_length[:2]

[{'page_number': -39,
  'sentence_chunk': 'Human Nutrition: 2020 Edition UNIVERSITY OF HAWAI‘I AT MĀNOA FOOD SCIENCE AND HUMAN NUTRITION PROGRAM ALAN TITCHENAL, SKYLAR HARA, NOEMI ARCEO CAACBAY, WILLIAM MEINKE-LAU, YA-YUN YANG, MARIE KAINOA FIALKOWSKI REVILLA, JENNIFER DRAPER, GEMADY LANGFELDER, CHERYL GIBBY, CHYNA NICOLE CHUN, AND ALLISON CALABRESE',
  'chunk_char_count': 308,
  'chunk_word_count': 42,
  'chunk_token_count': 77.0},
 {'page_number': -38,
  'sentence_chunk': 'Human Nutrition: 2020 Edition by University of Hawai‘i at Mānoa Food Science and Human Nutrition Program is licensed under a Creative Commons Attribution 4.0 International License, except where otherwise noted.',
  'chunk_char_count': 210,
  'chunk_word_count': 30,
  'chunk_token_count': 52.5}]

In [60]:
from sentence_transformers import util,SentenceTransformer
embedding_model=SentenceTransformer("all-mpnet-base-v2",
                                    device="cuda")
sentences=[
    "The Sentence Transformers library provides an easy and open-source way to create embeddings",
    "Sentences can be embedded one by one or as a list of strings.",
    "Embeddings are one of the most powerful concepts in machine learning",
    "Learn to use embeddings well and you'll be well on your way to being an AI engineer"
]

embeddings=embedding_model.encode(sentences)
embeddings_dict=dict(zip(sentences,embeddings))

for sentence, embedding in embeddings_dict.items():
  print("sentence",sentence)
  print("embedding",embedding)

sentence The Sentence Transformers library provides an easy and open-source way to create embeddings
embedding [-1.75474789e-02  4.15880829e-02 -1.96108501e-02  6.13632612e-02
 -2.26928238e-02 -1.66768041e-02 -9.68570448e-03 -5.56470007e-02
  6.51024468e-03 -2.71630790e-02  2.80722547e-02  3.37250829e-02
 -4.80537713e-02  2.39607915e-02  3.60239744e-02 -5.60144596e-02
  4.24363129e-02  1.26625551e-03 -2.09538117e-02  1.10234870e-02
  4.77666818e-02  3.43373306e-02  1.75942741e-02  5.75982258e-02
 -2.54101139e-02 -3.65637541e-02  3.65003385e-03 -2.21035257e-02
  4.46383357e-02  7.69358175e-03 -1.37073770e-02  3.51971132e-04
  3.10432799e-02  1.39934588e-02  8.64935146e-07 -8.51563737e-03
 -2.13634800e-02 -1.73759728e-03  7.27745425e-03  1.53444207e-03
  5.69876656e-02 -5.92064857e-02  1.12987654e-02  4.55607809e-02
 -3.81701663e-02 -8.00150726e-03  4.56678495e-02  2.50914600e-02
  8.72749984e-02  6.38238117e-02 -1.80346202e-02 -3.79043743e-02
  3.39744403e-03 -1.27106411e-02 -3.81009281

In [61]:
text_chunks=[item["sentence_chunk"] for item in pages_and_chunks_over_min_length]

In [62]:
text_chunk_embeddings=embedding_model.encode(text_chunks, batch_size=32, convert_to_tensor=True)
print(text_chunk_embeddings)

tensor([[ 0.0674,  0.0902, -0.0051,  ..., -0.0221, -0.0232,  0.0126],
        [ 0.0552,  0.0592, -0.0166,  ..., -0.0120, -0.0103,  0.0227],
        [ 0.0280,  0.0340, -0.0206,  ..., -0.0054,  0.0213,  0.0313],
        ...,
        [ 0.0771,  0.0098, -0.0122,  ..., -0.0409, -0.0752, -0.0241],
        [ 0.1030, -0.0165,  0.0083,  ..., -0.0574, -0.0283, -0.0295],
        [ 0.0864, -0.0125, -0.0113,  ..., -0.0522, -0.0337, -0.0299]],
       device='cuda:0')


In [63]:
print(len(text_chunk_embeddings))

1680


In [64]:
text_chunks_and_embeddings_df=pd.DataFrame(pages_and_chunks_over_min_length)
text_chunks_and_embeddings_df["embedding"] = (text_chunk_embeddings.cpu().numpy().tolist())
embeddings_df_save_path="text_chunks_and_embeddings_df.csv"
text_chunks_and_embeddings_df.to_csv(embeddings_df_save_path,index=False)


In [65]:
text_chunks_and_embedding_df_load=pd.read_csv(embeddings_df_save_path)
text_chunks_and_embedding_df_load.head()

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding
0,-39,Human Nutrition: 2020 Edition UNIVERSITY OF HA...,308,42,77.00,"[0.06742427498102188, 0.09022818505764008, -0...."
1,-38,Human Nutrition: 2020 Edition by University of...,210,30,52.50,"[0.055215608328580856, 0.059213943779468536, -..."
2,-37,Contents Preface University of Hawai‘i at Māno...,765,113,191.25,"[0.027980171144008636, 0.033981431275606155, -..."
3,-36,Lifestyles and Nutrition University of Hawai‘i...,940,141,235.00,"[0.0682566910982132, 0.03812750428915024, -0.0..."
4,-35,The Cardiovascular System University of Hawai‘...,998,152,249.50,"[0.03302641957998276, -0.008497655391693115, 0..."


In [66]:
df = pd.read_csv("text_chunks_and_embeddings_df.csv")

print(type(df["embedding"].iloc[0]))
print(df["embedding"].iloc[0][:200])

<class 'str'>
[0.06742427498102188, 0.09022818505764008, -0.0050954981707036495, -0.03175456449389458, 0.07390821725130081, 0.03519761934876442, -0.019798677414655685, 0.04676926136016846, 0.053572703152894974, 0.0


In [67]:
import ast
import numpy as np

text_chunks_and_embeddings_df = pd.read_csv("text_chunks_and_embeddings_df.csv")
text_chunks_and_embeddings_df["embedding"] = (text_chunks_and_embeddings_df["embedding"].apply(ast.literal_eval).apply(np.array))

pages_and_chunks = text_chunks_and_embeddings_df.to_dict(orient="records")

embeddings = torch.tensor(np.stack(text_chunks_and_embeddings_df["embedding"].values),dtype=torch.float32,device=device)

print(embeddings.shape)

torch.Size([1680, 768])


In [68]:
text_chunks_and_embeddings_df.head()

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding
0,-39,Human Nutrition: 2020 Edition UNIVERSITY OF HA...,308,42,77.00,"[0.06742427498102188, 0.09022818505764008, -0...."
1,-38,Human Nutrition: 2020 Edition by University of...,210,30,52.50,"[0.055215608328580856, 0.059213943779468536, -..."
2,-37,Contents Preface University of Hawai‘i at Māno...,765,113,191.25,"[0.027980171144008636, 0.033981431275606155, -..."
3,-36,Lifestyles and Nutrition University of Hawai‘i...,940,141,235.00,"[0.0682566910982132, 0.03812750428915024, -0.0..."
4,-35,The Cardiovascular System University of Hawai‘...,998,152,249.50,"[0.03302641957998276, -0.008497655391693115, 0..."


In [71]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-mpnet-base-v2",device="cuda")
query="macronutrients functions"
print(f"query:{query}")

query_embedding=embedding_model.encode(query, convert_to_tensor=True).to(device)

dot_scores=util.dot_score(query_embedding,embeddings)[0]

top_results_dot_product=torch.topk(dot_scores,k=5)
print(top_results_dot_product)

query:macronutrients functions
torch.return_types.topk(
values=tensor([0.6926, 0.6738, 0.6646, 0.6536, 0.6473], device='cuda:0'),
indices=tensor([42, 47, 41, 51, 46], device='cuda:0'))


In [72]:
import textwrap

def print_wrapped(text,wrap_length=80):
  wrapped_text=textwrap.fill(text,wrap_length)
  print(wrapped_text)

In [73]:
print(f"query {query}")
print("results:")
for score,idx in zip(top_results_dot_product[0],top_results_dot_product[1]):
  print(f"score:{score:.4f}")
  print("text:")
  print_wrapped(pages_and_chunks[idx]["sentence_chunk"])
  print(f"page number:{pages_and_chunks[idx]["page_number"]}")
  print("\n")

query macronutrients functions
results:
score:0.6926
text:
Macronutrients Nutrients that are needed in large amounts are called
macronutrients.There are three classes of macronutrients: carbohydrates, lipids,
and proteins.These can be metabolically processed into cellular energy.The
energy from macronutrients comes from their chemical bonds.This chemical energy
is converted into cellular energy that is then utilized to perform work,
allowing our bodies to conduct their basic functions.A unit of measurement of
food energy is the calorie.On nutrition food labels the amount given for
“calories” is actually equivalent to each calorie multiplied by one thousand.A
kilocalorie (one thousand calories, denoted with a small “c”) is synonymous with
the “Calorie” (with a capital “C”) on nutrition food labels.Water is also a
macronutrient in the sense that you require a large amount of it, but unlike the
other macronutrients, it does not yield calories. Carbohydrates Carbohydrates
are molecules com

In [93]:
def retrieve(query,embeddings,n_resources_return,model=embedding_model,):
  query_embedding=model.encode(query,convert_to_tensor=True)
  dot_scores=util.dot_score(query_embedding,embeddings)[0]

  scores,indices=torch.topk(input=dot_scores, k=n_resources_return)

  return scores,indices

In [75]:
query="symptoms of pellagra"
scores,indices=retrieve(query=query,embeddings=embeddings,n_resources_return=5)
print(scores)
print(f"indices: {indices}")


tensor([0.5000, 0.3741, 0.2959, 0.2793, 0.2721], device='cuda:0')
indices: tensor([ 822,  853, 1536, 1555, 1531], device='cuda:0')


In [76]:
from huggingface_hub import login
login(token="")

In [78]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import is_flash_attn_2_available
from transformers import BitsAndBytesConfig

quantization_config=BitsAndBytesConfig(load_in_4bit=True,
                                       bnb_4bit_compute_dtype=torch.float16)

attn_implementation="sdpa"
model_id = "google/gemma-2-2b-it"
tokenizer=AutoTokenizer.from_pretrained(model_id)
llm_model=AutoModelForCausalLM.from_pretrained(model_id,
                                               torch_dtype=torch.float16,
                                               quantization_config=quantization_config,
                                               low_cpu_mem_usage=False,
                                               attn_implementation=attn_implementation)

use_quantization_config = True
if not use_quantization_config:
  llm_model.to("cuda")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

In [79]:
input_text="what are the macronutrients, and what roles do they play in the human body?"
print(f"input:{input_text}")
prompt_template=[
    {"role":"user",
     "content":input_text}
]

prompt=tokenizer.apply_chat_template(conversation=prompt_template,
                                     tokenize=False,
                                     add_generation_prompt=True)

print(f"prompt formatted :{prompt}")

input:what are the macronutrients, and what roles do they play in the human body?
prompt formatted :<bos><start_of_turn>user
what are the macronutrients, and what roles do they play in the human body?<end_of_turn>
<start_of_turn>model



In [81]:
input_ids=tokenizer(prompt, return_tensors="pt").to("cuda")
print(f"input ids{input_ids}")
outputs=llm_model.generate(**input_ids,
                           max_new_tokens=256)
print(f"model output :{outputs[0]}\n")

input ids{'input_ids': tensor([[     2,      2,    106,   1645,    108,   5049,    708,    573, 186809,
         184592, 235269,    578,   1212,  16065,    749,    984,   1554,    575,
            573,   3515,   2971, 235336,    107,    108,    106,   2516,    108]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1]], device='cuda:0')}
model output :tensor([     2,      2,    106,   1645,    108,   5049,    708,    573, 186809,
        184592, 235269,    578,   1212,  16065,    749,    984,   1554,    575,
           573,   3515,   2971, 235336,    107,    108,    106,   2516,    108,
          4858, 235303, 235256,    476,  25497,    576, 186809, 184592,    578,
          1024,  16065,    575,    573,   3515,   2971, 235292,    109,    688,
         12298,   1695, 184592,    688,    708,  37132,    674,   1167,  13266,
          1476,    575,   2910,  15992, 235265,   2365,   3658,   4134,    5

In [82]:
outputs_decoded=tokenizer.decode(outputs[0])
print(f"decoded model output:{outputs_decoded}\n")

decoded model output:<bos><bos><start_of_turn>user
what are the macronutrients, and what roles do they play in the human body?<end_of_turn>
<start_of_turn>model
Here's a breakdown of macronutrients and their roles in the human body:

**Macronutrients** are nutrients that our bodies need in large amounts. They provide energy and are essential for various bodily functions. 

**The three main macronutrients are:**

* **Carbohydrates:**
    * **Role:** Primary source of energy for the body. They break down into glucose, which fuels cells and tissues.
    * **Types:**
        * **Simple carbohydrates:** Sugars like glucose, fructose, and lactose. Found in fruits, milk, and processed foods.
        * **Complex carbohydrates:** Starches and fiber. Found in whole grains, vegetables, and legumes.
    * **Benefits:** Provide energy, support brain function, and aid in digestion.

* **Proteins:**
    * **Role:** Building blocks of tissues, enzymes, hormones, and other essential molecules. They als

In [83]:
print(f"input text:{input_text}")
print(f"output text: {outputs_decoded.replace(prompt,'').replace('<bos>','').replace('<eos>','')}")

input text:what are the macronutrients, and what roles do they play in the human body?
output text: Here's a breakdown of macronutrients and their roles in the human body:

**Macronutrients** are nutrients that our bodies need in large amounts. They provide energy and are essential for various bodily functions. 

**The three main macronutrients are:**

* **Carbohydrates:**
    * **Role:** Primary source of energy for the body. They break down into glucose, which fuels cells and tissues.
    * **Types:**
        * **Simple carbohydrates:** Sugars like glucose, fructose, and lactose. Found in fruits, milk, and processed foods.
        * **Complex carbohydrates:** Starches and fiber. Found in whole grains, vegetables, and legumes.
    * **Benefits:** Provide energy, support brain function, and aid in digestion.

* **Proteins:**
    * **Role:** Building blocks of tissues, enzymes, hormones, and other essential molecules. They also play a role in immune function, wound healing, and muscle g

In [84]:
questions=[
    "How often should infants be breastfed?",
    "What are the symptoms of pellagra?",
    "What are macronutrients and what roles do they play in the human body?",
    "What is the RDI for protein per day?",
    "How does saliva help with digestion?",
    "water soluble vitamins"
]

query_list=questions

In [85]:
def prompt_formatter(query,context_items):
  context="- "+"\n-".join([item["sentence_chunk"] for item in context_items])
  base_prompt="""based on the following context items, please answer the query.
  Give yourself room to think by extracting relevant passages from the context before answering the query.
  Make sure the answers are as explantory as possible.
  Use the following examples as refrence for the ideal answer style.
  \n Example 1:
  Query:What are fat-soluble vitamins?
  Answer: The fat-soluble vitamins include Vitamin A, Vitamin D, VItamin E, and VItamin K. These vitamins are absorbed along with the fats in the diet.
  \n Example 2:
  Query: What are the causes of type 2 diabetes?
  Answer: Type 2 diabetes is often associated with overnutrition, particularly the overconsumption of calories leading to obesity.
  \n Example 3:
  Query: What is the importance of hydration for physical performance?
  Answer: Hydration is crucial for physical perfprmance because water plays key roles iin maintaining blood volume, regulationg body temperature, and ensuring better health.
  \nNow use the following context items to answer the use query:
  {context}. The context contains the answer. Look for relevant information.
  \nRelevant passages: <extract relevant passages from the context here>
  User query:{query}
  Answer: """

  base_prompt=base_prompt.format(context=context, query=query)
  prompt_template=[
    {"role":"user",
     "content":input_text}
]

  prompt=tokenizer.apply_chat_template(conversation=prompt_template,
                                     tokenize=False,
                                     add_generation_prompt=True)

  return prompt



In [94]:
query=random.choice(query_list)
print(f"query:{query}")
scores,indices=retrieve(query=query,embeddings=embeddings,n_resources_return=5)
context_items=[pages_and_chunks[i] for i in indices]
prompt=prompt_formatter(query=query, context_items=context_items)
print(prompt)

query:What are the symptoms of pellagra?
<bos><start_of_turn>user
what are the macronutrients, and what roles do they play in the human body?<end_of_turn>
<start_of_turn>model



In [95]:
input_ids=tokenizer(prompt, return_tensors="pt").to("cuda")
outputs=llm_model.generate(**input_ids,
                           temperature=0.7,
                           do_sample=True,
                           max_new_tokens=256)

output_text=tokenizer.decode(outputs[0])
print(f"query:{query}")
print(f"answer: \n{output_text.replace(prompt,'')}")

query:What are the symptoms of pellagra?
answer: 
<bos>## Macronutrients: Fueling Your Body's Engine

Macronutrients are the nutrients our bodies need in large quantities to function properly and thrive. They provide the building blocks for growth, energy, and various bodily processes. These are the three main types:

**1. Carbohydrates:**

* **Role:** 
    * Primary source of energy for the body.
    * Broken down into glucose, which is used by cells for energy production.
    * Supports brain function, muscle contraction, and blood sugar regulation.
* **Types:** 
    * Simple carbohydrates (sugars) like glucose and fructose: quick source of energy, found in fruits, honey, and processed foods.
    * Complex carbohydrates (starches and fiber): longer-lasting energy, found in whole grains, vegetables, and legumes.

**2. Protein:**

* **Role:** 
    * Crucial for building and repairing tissues, including muscles, bones, skin, and hair.
    * Helps in hormone production, immune function, 